# Calibrating the empirical background width

This tutorial compares the legacy $a=0.66$, $b=0.88$ smoothing law with nearby alternatives. Detection and $\Delta\nu$ recovery are kept as separate objectives, and the final comparison uses a held-out observing window.

In [ ]:
import numpy as np

from urdr import (
    BackgroundBenchmarkCase,
    EmpiricalGridPoint,
    SimulationConfig,
    benchmark_empirical_grid,
    make_observing_window,
)

## Define training and validation cases

The example is deliberately small enough to execute quickly. A production run should span more $\nu_{\max}$ values, oscillation amplitudes, durations, and real target masks, with at least 128 realisations per condition.

In [ ]:
def make_case(name, split, gap, random_missing, seed):
    window = make_observing_window(
        duration_days=1.0,
        cadence_seconds=120.0,
        gaps_days=(gap,),
        random_missing_fraction=random_missing,
        seed=seed,
    )
    simulation = SimulationConfig(
        white_noise_sigma=0.3,
        granulation_amplitude=0.2,
        granulation_timescale_days=0.15,
        numax_uhz=1000.0,
        delta_nu_uhz=100.0,
        envelope_width_uhz=350.0,
        oscillation_amplitude=1.0,
    )
    return BackgroundBenchmarkCase(
        name=name,
        split=split,
        window=window,
        simulation=simulation,
        centre_frequencies_uhz=np.array([900.0, 1000.0, 1100.0]),
        filter_width_uhz=500.0,
        delta_nu_grid_uhz=np.linspace(90.0, 110.0, 5),
        oscillation_amplitudes=np.array([0.4, 1.0]),
    )

cases = [
    make_case("training_gap", "train", (0.40, 0.45), 0.01, 1),
    make_case("held_out_gap", "validation", (0.65, 0.75), 0.05, 2),
]
[(case.name, case.window.duty_cycle) for case in cases]

## Run paired simulations

All candidates see the same random realisations within each case. The low realisation count here is only for tutorial speed.

In [ ]:
candidates = [
    EmpiricalGridPoint(0.45, 0.80),
    EmpiricalGridPoint(0.66, 0.88),
    EmpiricalGridPoint(0.90, 0.95),
]
result = benchmark_empirical_grid(
    cases,
    candidates,
    anchors=24,
    minimum_bins=3,
    realizations=8,
    target_false_positive_rate=0.25,
    max_lag_seconds=15_000.0,
    seed=42,
)

## Keep the two scientific objectives visible

The Pareto frontier contains candidates for which no other candidate improves both mean detection rate and mean $\Delta\nu$ recovery. Validation summaries then show whether those choices generalise to the unseen window.

In [ ]:
training_front = result.pareto_front("train")
validation = result.summaries("validation")
training_front, validation

## Export machine-readable rows

`to_records` avoids a pandas dependency. If pandas is available, `pandas.DataFrame(result.to_records())` can be written directly to CSV.

In [ ]:
records = result.to_records()
records[:2]